Run all when generating a PDB, as the code overrides positions compiled, so it has to compile the initial positions again, otherwise you would be over twisting.

In [47]:
import MDAnalysis as md
import numpy as np
import twistaly as tw

filedir = "./"
outputdir = "twisted/"

fib = md.Universe(filedir + "colfib.pdb")

In [48]:
prot = fib.select_atoms("protein")
pos = prot.positions / 10
z_middle = (max(pos[:,2]) + min(pos[:,2]))/2

# Center
com = prot.center_of_mass() / 10

# Transformation function
m_psi = 0.07 #phi/z -> psi = tan^-1(r*m) (dependance on radius)
twist = lambda z: m_psi*(z - z_middle - min(pos[:,2])) # so that in the middle is the shifting point of translation, ie, twisted induced

rad, phi, z = tw.cyl_proj(pos, x_c=com[0], y_c=com[1])

phi_twisted = phi + twist(z)

x_tw, y_tw = rad*np.cos(phi_twisted) + com[0], rad*np.sin(phi_twisted) + com[1]

In [49]:
new_pos = np.column_stack([x_tw, y_tw, z]).astype(np.float64)

# Write back into the Universe (protein only)
prot.positions = new_pos * 10

# Export whole system with protein twisted and everything else unchanged
fib.atoms.write(outputdir + f"colfib-tw{m_psi}.pdb")
print("Wrote PDB")

Wrote PDB
